In [10]:
%gui qt6
%matplotlib inline
%matplotlib widget
import mne
import os
from data import eeg
from analysis import isc
from analysis import plotting
import numpy as np
from pathlib import Path
from ipywidgets import Layout, widgets
from IPython.display import display
from matplotlib import pyplot as plt


working_dir = Path(os.getcwd())
if working_dir.name == "notebooks":
    working_dir = working_dir.parent

print("Working directory:", working_dir)

data_dir = Path(os.getenv("EEG_WORK_DIR", "./out"))
if not data_dir.is_absolute():
    data_dir = working_dir / data_dir


Working directory: /Users/au662726/github_projects/byd-hyperscanning-analysis


In [3]:

all_data = eeg.load_all_eeg(data_dir, preprocessed=True)


Loading BangBangYouAreDead: 100%|██████████| 10/10 [00:00<00:00, 15.28it/s]


# Data Exploration

In [ ]:
plt.ioff()

# ── Widgets ────────────────────────────────────────────────────────────────────
initial_stimulus = eeg.STIMULI[1]
initial_subject_id = eeg.SUBJECT_IDS[0]
initial_raw = all_data[initial_stimulus][initial_subject_id]
initial_duration = initial_raw.n_times / initial_raw.info["sfreq"]

stim_select = widgets.Dropdown(
    options=[(s, s) for s in eeg.STIMULI],
    description="Stimulus:",
    value=initial_stimulus,
    layout=Layout(width="auto"),
)

subject_select = widgets.Dropdown(
    options=[(f"Subject {sid:02d}", sid) for sid in eeg.SUBJECT_IDS],
    description="Subject:",
    value=initial_subject_id,
    layout=Layout(width="auto"),
)

channel_selection = widgets.SelectMultiple(
    options=initial_raw.ch_names,
    value=["Cz", "P3", "C3"],
    description="Channels:",
    rows=12,
    layout=Layout(width="auto"),
)

time_start = widgets.BoundedFloatText(
    value=20.0, min=0.0, max=initial_duration,
    step=1.0, description="From (s):",
    style={"description_width": "initial"},
    layout=Layout(width="180px"),
)

time_end = widgets.BoundedFloatText(
    value=100.0, min=0.0, max=initial_duration,
    step=1.0, description="To (s):",
    style={"description_width": "initial"},
    layout=Layout(width="180px"),
)

plot_button = widgets.Button(
    description="Plot EEG",
    button_style="primary",
    icon="eye",
    layout=Layout(width="150px", height="36px"),
)

eeg_output = widgets.Output()

# ── Layout ─────────────────────────────────────────────────────────────────────
left_panel = widgets.VBox([stim_select, subject_select, channel_selection])
right_panel = widgets.VBox([
    widgets.HBox([time_start, time_end]),
    plot_button,
])
controls = widgets.HBox([left_panel, widgets.Box(layout=Layout(width="30px")), right_panel])

display(controls, eeg_output)


# ── Callback ───────────────────────────────────────────────────────────────────
def plot_eeg_data(_):
    eeg_output.clear_output(wait=True)
    with eeg_output:
        stimulus = stim_select.value
        subject_id = subject_select.value
        selected_channels = list(channel_selection.value)
        t_start = time_start.value
        t_end = time_end.value

        if t_end <= t_start:
            print("'To' must be greater than 'From'.")
            return

        raw = all_data[stimulus][subject_id]
        display(f"Showing data for Subject {subject_id:02d} - {stimulus}")
        display(f"Sample rate: {raw.info['sfreq']} Hz, Total duration: {raw.n_times / raw.info['sfreq']:.2f} seconds")
        display(f"Time window: {t_start:.2f} - {t_end:.2f} seconds")

        picks = mne.pick_channels(raw.info["ch_names"], include=selected_channels)

        eeg_plot = raw.plot(
            start=t_start,
            duration=t_end - t_start,
            picks=picks,
            use_opengl=True, block=True, show_scrollbars=False,
            overview_mode="hidden", splash=False, verbose="error", show=False,
        )
        out = widgets.Output()
        out.layout = Layout(width="100%", height="auto", margin="10px 0")
        with out:
            plt.show(eeg_plot)
        display(out)


plot_button.on_click(plot_eeg_data)


Output()

# Inter-Subject Correlation (ISC) Analysis

Pick the stimulus, subjects and time windows to include in ISC, then the ISC parameters.

In [ ]:
plt.ioff()

isc_results = {}  # populated by run_isc, consumed by subsequent cells

# ── Widgets ────────────────────────────────────────────────────────────────────
isc_stim_select = widgets.Dropdown(
    options=[(s, s) for s in eeg.STIMULI],
    value=eeg.STIMULI[0],
    description="Stimulus:",
    layout=Layout(width="300px"),
)

isc_subject_select = widgets.SelectMultiple(
    options=[(f"Subject {sid:02d}", sid) for sid in eeg.SUBJECT_IDS],
    value=list(eeg.SUBJECT_IDS),
    description="Subjects:",
    rows=len(eeg.SUBJECT_IDS),
    layout=Layout(width="200px"),
)

isc_window_slider = widgets.FloatSlider(
    value=5.0, min=1.0, max=30.0, step=0.5,
    description="Window (s):",
    style={"description_width": "initial"},
    layout=Layout(width="400px"),
)

isc_step_slider = widgets.FloatSlider(
    value=1.0, min=0.5, max=10.0, step=0.5,
    description="Step (s):",
    style={"description_width": "initial"},
    layout=Layout(width="400px"),
)

n_components_slider = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description="Components:",
    style={"description_width": "initial"},
    layout=Layout(width="400px"),
)

run_button = widgets.Button(
    description="Run ISC",
    button_style="primary",
    icon="play",
    layout=Layout(width="150px", height="36px"),
)

isc_output = widgets.Output()

left_panel = widgets.VBox([isc_stim_select, isc_subject_select])
right_panel = widgets.VBox([isc_window_slider, isc_step_slider, n_components_slider, run_button])
controls = widgets.HBox([left_panel, widgets.Box(layout=Layout(width="30px")), right_panel])

display(controls, isc_output)


def run_isc(_):
    isc_output.clear_output(wait=True)
    with isc_output:
        stimulus = isc_stim_select.value
        subject_ids = list(isc_subject_select.value)
        window_sec = isc_window_slider.value
        step_sec = isc_step_slider.value
        n_comp = n_components_slider.value

        if len(subject_ids) < 2:
            print("Please select at least 2 subjects.")
            return

        arrays = [all_data[stimulus][s].get_data() for s in subject_ids]
        min_t = min(a.shape[1] for a in arrays)
        data_array = np.array([a[:, :min_t] for a in arrays])

        fs = int(all_data[stimulus][subject_ids[0]].info["sfreq"])

        print(f"Stimulus : {stimulus}")
        print(f"Subjects : {subject_ids}  (N={len(subject_ids)}, T={min_t} samples @ {fs} Hz = {min_t/fs:.1f}s)")
        print(f"Window   : {window_sec}s  |  Step: {step_sec}s  |  Components: {n_comp}")

        W, ISC_train = isc.train_cca({stimulus: data_array})
        ISC, ISC_persecond, ISC_bysubject, A, window_times = isc.apply_cca(
            data_array, W, fs, window_sec=window_sec, step_sec=step_sec
        )

        n_comp = min(n_comp, ISC.shape[0])

        # Store for downstream cells
        isc_results.update({
            "W": W, "A": A, "ISC": ISC,
            "ISC_persecond": ISC_persecond,
            "ISC_bysubject": ISC_bysubject,
            "window_times": window_times,
            "stimulus": stimulus,
            "subject_ids": subject_ids,
            "fs": fs,
            "n_comp": n_comp,
            "info": all_data[stimulus][subject_ids[0]].info,
            "data_array": data_array,
        })

        fig, axes = plt.subplots(2, 1, figsize=(13, 7), gridspec_kw={"height_ratios": [3, 1]})

        for c in range(n_comp):
            axes[0].plot(window_times, ISC_persecond[c],
                         label=f"Comp {c + 1}  (global ISC = {ISC[c]:.3f})", linewidth=1.2)
        axes[0].axhline(0, color="black", linewidth=0.6, linestyle="--")
        axes[0].set_xlabel("Time (s)")
        axes[0].set_ylabel("ISC")
        axes[0].set_title(f"ISC per window — {stimulus}  (N={len(subject_ids)}, window={window_sec}s, step={step_sec}s)")
        axes[0].legend(loc="upper right", fontsize=8)
        axes[0].grid(True, alpha=0.3)

        heatmap_data = ISC_bysubject[:n_comp, :]
        abs_max = np.abs(heatmap_data).max()
        im = axes[1].imshow(heatmap_data, aspect="auto", cmap="RdBu_r", vmin=-abs_max, vmax=abs_max)
        axes[1].set_yticks(range(n_comp))
        axes[1].set_yticklabels([f"Comp {c + 1}" for c in range(n_comp)], fontsize=8)
        axes[1].set_xticks(range(len(subject_ids)))
        axes[1].set_xticklabels([f"S{sid:02d}" for sid in subject_ids], fontsize=8)
        axes[1].set_title(f"ISC by subject (leave-one-out pairwise, range [{-abs_max:.3f}, {abs_max:.3f}])")
        plt.colorbar(im, ax=axes[1], orientation="vertical", fraction=0.02, pad=0.01)

        plt.tight_layout()
        plt.show()


run_button.on_click(run_isc)


Output()

## Component Inspection
Scalp topographies (from scalp projection matrix **A**) and spatial filter weights (**W**) for the components returned by CCA.

In [13]:
plt.ioff()

topo_button = widgets.Button(
    description="Inspect Components",
    button_style="info",
    icon="search",
    layout=Layout(width="200px", height="36px"),
)
topo_output = widgets.Output()

display(topo_button, topo_output)


def inspect_components(_):
    topo_output.clear_output(wait=True)
    with topo_output:
        if not isc_results:
            print("Run ISC first.")
            return

        A = isc_results["A"]
        W = isc_results["W"]
        n_comp = isc_results["n_comp"]
        info = isc_results["info"]
        ch_names = info["ch_names"]

        try:
            cmap = parula_map  # use if defined in the session
        except NameError:
            cmap = "RdBu_r"

        # ── Scalp topographies ─────────────────────────────────────────────────
        fig, axes = plt.subplots(1, n_comp, figsize=(3.2 * n_comp, 3.5))
        if n_comp == 1:
            axes = [axes]

        for comp_i in range(n_comp):
            mne.viz.plot_topomap(
                A[:, comp_i],
                info,
                cmap=cmap,
                show=False,
                contours=6,
                axes=axes[comp_i],
                sphere=0.09,
                extrapolate="head",
                vlim=(A[:, comp_i].min(), A[:, comp_i].max()),
            )
            axes[comp_i].set_title(
                f"Comp {comp_i + 1}\nISC = {isc_results['ISC'][comp_i]:.3f}", fontsize=9
            )

        fig.suptitle(f"Scalp projections (A) — {isc_results['stimulus']}", fontsize=11)
        plt.tight_layout()
        plt.show()

        # ── Spatial filter weight matrix (W) ───────────────────────────────────
        # Normalize each column to unit norm — the absolute scale of CCA filters
        # is arbitrary; only the relative weights across channels matter
        W_display = W[:, :n_comp]
        W_display = W_display / np.linalg.norm(W_display, axis=0, keepdims=True)
        abs_max = np.abs(W_display).max()

        fig2, ax = plt.subplots(figsize=(max(5, n_comp * 0.9), max(4, len(ch_names) * 0.35)))
        im = ax.imshow(W_display, aspect="auto", cmap="RdBu_r", vmin=-abs_max, vmax=abs_max)
        ax.set_yticks(range(len(ch_names)))
        ax.set_yticklabels(ch_names, fontsize=8)
        ax.set_xticks(range(n_comp))
        ax.set_xticklabels([f"Comp {c + 1}" for c in range(n_comp)], fontsize=9)
        ax.set_title("Spatial filters (W, unit-norm per component) — rows are channels", fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        plt.tight_layout()
        plt.show()


topo_button.on_click(inspect_components)


Button(button_style='info', description='Inspect Components', icon='search', layout=Layout(height='36px', widt…

Output()

## Apply Spatial Filter
Project the EEG data through the selected CCA component to produce per-subject component timecourses.
The result is stored in `component_data` as a `(subjects × samples)` numpy array.

In [ ]:
plt.ioff()

comp_select = widgets.BoundedIntText(
    value=1, min=1, max=10, step=1,
    description="Component:",
    style={"description_width": "initial"},
    layout=Layout(width="180px"),
)

apply_button = widgets.Button(
    description="Apply & Plot",
    button_style="success",
    icon="check",
    layout=Layout(width="150px", height="36px"),
)

apply_output = widgets.Output()

display(widgets.HBox([comp_select, apply_button]), apply_output)

# Accessible from subsequent cells after clicking Apply
component_data = None


def apply_weights(_):
    global component_data
    apply_output.clear_output(wait=True)
    with apply_output:
        if not isc_results:
            print("Run ISC first.")
            return

        comp_i = comp_select.value - 1  # convert 1-indexed to 0-indexed
        n_comp = isc_results["n_comp"]

        if comp_i >= n_comp:
            print(f"Only {n_comp} component(s) available. Adjust the Component selector.")
            return

        W = isc_results["W"]
        data_array = isc_results["data_array"]   # (N, D, T)
        subject_ids = isc_results["subject_ids"]
        fs = isc_results["fs"]
        stimulus = isc_results["stimulus"]

        # Project: w[:, comp_i] is the spatial filter (D,); w^T @ X gives (T,) per subject
        spatial_filter = W[:, comp_i]  # (D,)
        component_data = np.array([spatial_filter @ data_array[i] for i in range(len(subject_ids))])  # (N, T)

        T = component_data.shape[1]
        t = np.arange(T) / fs

        print(f"Component {comp_select.value}  (ISC = {isc_results['ISC'][comp_i]:.4f})")
        print(f"Output shape : {component_data.shape}  (subjects × samples)")
        print("Stored in    : component_data")

        fig, ax = plt.subplots(figsize=(13, 4))
        for i, sid in enumerate(subject_ids):
            ax.plot(t, component_data[i], linewidth=0.6, alpha=0.7, label=f"S{sid:02d}")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude (a.u.)")
        ax.set_title(
            f"Component {comp_select.value} timecourses — {stimulus}  "
            f"(ISC = {isc_results['ISC'][comp_i]:.3f})"
        )
        ax.legend(loc="upper right", fontsize=7, ncol=2)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


apply_button.on_click(apply_weights)


Output()